# 02. Clustering and cell-type annotation

Input: `qc_annotated.h5ad` from notebook 01. Output: `annotated.h5ad` with cell types (`cell_type`), a doublet flag
(`exclude`), a cycling score, raw counts (`layers["counts"]`) and a correctly log-normalised layer (`layers["lognorm"]`, also set as `.X`).

**Reproducibility note.** The Leiden clusters used for annotation were generated on a matrix that had been normalised
and log-transformed twice (the clustering cell was re-run after installing `igraph`, and scanpy warned that `X` looked
already log-transformed). The annotation maps cluster numbers from that run to cell types, so this notebook reproduces
the double transform by default (`DOUBLE_TRANSFORM = True`). Everything downstream uses raw counts or the rebuilt
`lognorm` layer, not the clustering matrix. The last cell compares the clustering with one on single normalisation.

In [ ]:
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt

# ---- edit these two paths ----------------------------------------------------
data_dir = Path("data")                        # folder with the extracted GSE217930 files
out_dir = Path("E:/malaria_scrna_results")     # large intermediate files, needs about 5 GB free
# ------------------------------------------------------------------------------
fig_dir = out_dir / "figures"
tab_dir = out_dir / "tables"
for d in (out_dir, fig_dir, tab_dir):
    d.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = fig_dir
sc.set_figure_params(dpi=100, dpi_save=200, facecolor="white")
print({p: version(p) for p in ['scanpy', 'anndata', 'pandas', 'numpy']})


## Load

In [ ]:
adata = sc.read_h5ad(out_dir / "qc_annotated.h5ad")
print(adata)


## Normalise, reduce and cluster

In [ ]:
DOUBLE_TRANSFORM = True   # True reproduces the reported clustering; False = standard single normalisation

for _ in range(2 if DOUBLE_TRANSFORM else 1):
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=3000, batch_key="donor")
sc.pp.pca(adata, n_comps=30, mask_var="highly_variable")
sc.pp.neighbors(adata, n_pcs=30)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.6, flavor="igraph", n_iterations=2)

sc.pl.umap(adata, color=["leiden", "day", "batch", "donor"], wspace=0.4)
print(adata.obs["leiden"].nunique(), "clusters")


## Gene symbols as gene names
Ensembl IDs are kept in `var["ensembl_id"]`.

In [ ]:
# Keep the Ensembl IDs, then use symbols as the gene names
adata.var["ensembl_id"] = adata.var_names

symbols = adata.var["gene_symbol"].astype(str)
# Fall back to the Ensembl ID where the symbol is missing or empty
bad = symbols.isin(["nan", "", "None"])
symbols[bad] = adata.var_names[bad]

adata.var_names = symbols
adata.var_names_make_unique()

print(adata.var_names[:5].tolist())
print("genes without a symbol:", int(bad.sum()))
print([g for g in ["IL7R", "CD14", "NKG7", "MS4A1", "CD3E"] if g in adata.var_names])


## Marker genes per cluster

In [ ]:
markers = {
    "CD4 T":        ["IL7R", "CCR7", "LTB", "TCF7"],
    "Treg":         ["FOXP3", "IL2RA", "CTLA4"],
    "CD8 T":        ["CD8A", "CD8B", "GZMK", "CCL5"],
    "gd T":         ["TRDC", "TRGC1", "TRGC2"],
    "NK":           ["NKG7", "GNLY", "KLRD1", "NCAM1"],
    "B naive":      ["MS4A1", "CD79A", "IGHD", "TCL1A"],
    "B memory":     ["CD27", "AIM2"],
    "Plasma":       ["JCHAIN", "MZB1", "XBP1"],
    "CD14 mono":    ["CD14", "LYZ", "S100A8", "S100A9"],
    "CD16 mono":    ["FCGR3A", "LST1", "MS4A7"],
    "cDC":          ["CST3", "CD1C", "FCER1A"],
    "pDC":          ["LILRA4", "IRF7", "CLEC4C"],
    "Platelet":     ["PPBP", "PF4"],
    "Erythroid":    ["HBB", "HBA1"],
    "Cycling":      ["MKI67", "TOP2A"],
}
markers = {k: [g for g in v if g in adata.var_names] for k, v in markers.items()}

sc.pl.dotplot(adata, markers, groupby="leiden", standard_scale="var", dendrogram=False)

sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon")
top = pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).head(8)
top


In [ ]:
# Check ambiguous clusters (3, 4, 8): T-cell markers and mitochondrial content
sc.pl.dotplot(adata, ["CD3E", "CD4", "CD8A", "CD8B", "NCAM1", "FCGR3A", "KLRD1"],
              groupby="leiden", standard_scale="var",
              save="_check_T_vs_NK.png")

mt_col = [c for c in ["pct_counts_mt", "pct_mt"] if c in adata.obs]
if mt_col:
    print(adata.obs.groupby("leiden")[mt_col[0]].median().round(2))


## Doublet checks: clusters co-expressing B and T (or monocyte and T) markers

In [ ]:
import numpy as np

def coexpr(a, b):
    X = adata[:, [a, b]].X
    X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    both = (X[:, 0] > 0) & (X[:, 1] > 0)
    return pd.Series(both, index=adata.obs_names).groupby(
        adata.obs["leiden"].values, observed=True).mean()

co = pd.DataFrame({
    "MS4A1+CD3E": coexpr("MS4A1", "CD3E"),
    "LYZ+CD3E":   coexpr("LYZ", "CD3E"),
}).round(2)
print(co)

cols = [c for c in ["total_counts", "n_genes_by_counts"] if c in adata.obs]
print(adata.obs.groupby("leiden", observed=True)[cols].median())


In [ ]:
# Safety check: the annotation below maps Leiden cluster NUMBERS to cell types, so it is only valid
# if the clustering came out the same as in the reported analysis. Stop here if it did not.
assert adata.obs["leiden"].nunique() == 18, (
    "Expected 18 Leiden clusters, got %d. Cluster numbers differ from the reported run, "
    "so the `fine` mapping would mislabel cells. Re-annotate from the marker dotplot." % adata.obs["leiden"].nunique())
doublet_clusters = sorted(co.index[(co["MS4A1+CD3E"] > 0.5) | (co["LYZ+CD3E"] > 0.5)].astype(str))
assert doublet_clusters == ["11", "13"], (
    "B+T / monocyte+T co-expressing clusters are %s, expected ['11', '13']. "
    "Re-annotate from the marker dotplot." % doublet_clusters)
print("clustering matches the reported run: 18 clusters, doublet-like clusters 11 and 13")


## Annotation
Cluster numbers are mapped to cell subtypes and then to coarse cell types. Clusters co-expressing two lineages are flagged as doublet-like and excluded from all downstream tests.

In [ ]:
fine = {
    "0": "B naive",           "1": "CD4 T memory",       "2": "CD8 T naive",
    "3": "CD4 T (MT-high)",   "4": "T activated",        "5": "CD4 T naive",
    "6": "CD4 T naive",       "7": "CD8 T effector memory",
    "8": "CD8 T cytotoxic",   "9": "NK",                 "10": "Plasma",
    "11": "Doublet-like (B+T)",
    "12": "pDC",              "13": "Doublet-like (Mono+lymph)",
    "14": "CD16 mono",        "15": "CD14 mono",         "16": "CD14 mono",
    "17": "Platelet",
}
coarse = {
    "B naive": "B", "Plasma": "Plasma",
    "CD4 T memory": "CD4 T", "CD4 T naive": "CD4 T", "CD4 T (MT-high)": "CD4 T",
    "CD8 T naive": "CD8 T", "CD8 T effector memory": "CD8 T", "CD8 T cytotoxic": "CD8 T",
    "T activated": "T activated", "NK": "NK", "pDC": "pDC",
    "CD14 mono": "Monocyte", "CD16 mono": "Monocyte", "Platelet": "Platelet",
    "Doublet-like (B+T)": "Doublet-like",
    "Doublet-like (Mono+lymph)": "Doublet-like",
}

adata.obs["cell_subtype"] = adata.obs["leiden"].astype(str).map(fine).astype("category")
adata.obs["cell_type"] = adata.obs["cell_subtype"].map(coarse).astype("category")
print(adata.obs["cell_type"].value_counts())

sc.pl.umap(adata, color=["cell_type", "cell_subtype"], legend_loc="on data",
           legend_fontsize=7, save="_cell_types.png")

adata.obs["exclude"] = adata.obs["cell_type"].eq("Doublet-like")
print(adata.obs["exclude"].value_counts())


## Dendritic cells
There is no separate cDC cluster at this resolution, so monocytes are sub-clustered to look for cDC2.

In [ ]:
import numpy as np
import pandas as pd

dc = [g for g in ["CD1C", "FCER1A", "CLEC10A", "CLEC9A", "HLA-DQA1"] if g in adata.var_names]
print("present:", dc)

sc.pl.dotplot(adata, dc, groupby="leiden", standard_scale="var")

X = adata[:, ["CD1C", "FCER1A"]].X
X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
both = pd.Series((X[:, 0] > 0) & (X[:, 1] > 0), index=adata.obs_names)
print(both.groupby(adata.obs["leiden"].values, observed=True).mean().round(3))


In [ ]:
# 1) Is adata.X normalised once, and is the counts layer intact?
print("layers:", list(adata.layers.keys()))
C = adata.layers["counts"][:2000]
C = C.toarray() if hasattr(C, "toarray") else np.asarray(C)
print("counts integer:", np.allclose(C, np.round(C)), "| max:", C.max())

Xs = adata.X[:2000]
Xs = Xs.toarray() if hasattr(Xs, "toarray") else np.asarray(Xs)
expected = np.log1p(C / C.sum(axis=1, keepdims=True) * 1e4)
print("X == log1p(normalised counts):", np.allclose(Xs, expected, atol=1e-4))

# 2) Sub-cluster the monocytes to look for cDCs
mono = adata[adata.obs["cell_type"] == "Monocyte"].copy()
sc.pp.neighbors(mono, n_pcs=30)
sc.tl.leiden(mono, resolution=0.5, key_added="mono_sub",
             flavor="igraph", n_iterations=2, directed=False)
print(mono.obs["mono_sub"].value_counts().sort_index())

sc.pl.dotplot(mono, ["CD1C", "FCER1A", "CLEC10A", "HLA-DQA1",
                     "CD14", "FCGR3A", "LYZ", "S100A8"],
              groupby="mono_sub", standard_scale="var", save="_cDC_subcluster.png")


In [ ]:
# Safety check: sub-cluster 3 of the monocytes should be the cDC2 population (CD1C + FCER1A)
def coexpr_mono(a, b):
    X = mono[:, [a, b]].X
    X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    both = pd.Series((X[:, 0] > 0) & (X[:, 1] > 0), index=mono.obs_names)
    return both.groupby(mono.obs["mono_sub"].values, observed=True).mean()

cdc2 = coexpr_mono("CD1C", "FCER1A")
print(cdc2.round(3).to_string())
assert cdc2.idxmax() == "3", ("Monocyte sub-cluster %s, not 3, has the highest CD1C+FCER1A "
                              "co-expression; relabel the cDC2 cells accordingly." % cdc2.idxmax())


## Label cDC2 and rebuild a correct log-normalised layer from raw counts

In [ ]:
import scipy.sparse as sp

# 1) Label the cDC2 cells found in monocyte sub-cluster 3
idx = mono.obs_names[mono.obs["mono_sub"] == "3"]
adata.obs["cell_subtype"] = adata.obs["cell_subtype"].astype(str)
adata.obs["cell_type"] = adata.obs["cell_type"].astype(str)
adata.obs.loc[idx, "cell_subtype"] = "cDC2"
adata.obs.loc[idx, "cell_type"] = "cDC"
adata.obs["cell_subtype"] = adata.obs["cell_subtype"].astype("category")
adata.obs["cell_type"] = adata.obs["cell_type"].astype("category")
print(adata.obs.loc[idx, "leiden"].value_counts())   # which Leiden clusters they came from
print(adata.obs["cell_type"].value_counts())

# 3) Rebuild a correct log-normalised layer from raw counts
tmp = sc.AnnData(sp.csr_matrix(adata.layers["counts"]).astype("float32"))
sc.pp.normalize_total(tmp, target_sum=1e4)
sc.pp.log1p(tmp)
adata.layers["lognorm"] = tmp.X
del tmp

C = adata.layers["counts"][:2000]
C = C.toarray() if hasattr(C, "toarray") else np.asarray(C)
Xl = adata.layers["lognorm"][:2000]
Xl = Xl.toarray() if hasattr(Xl, "toarray") else np.asarray(Xl)
print("lognorm correct:", np.allclose(Xl, np.log1p(C / C.sum(axis=1, keepdims=True) * 1e4), atol=1e-4))
print("old X max:", float(adata.X[:2000].max()), "| lognorm max:", float(Xl.max()))


In [ ]:
# Use the verified log-normalised layer as X
adata.X = adata.layers["lognorm"].copy()
print("X restored:", adata.X is not None, adata.X.shape)

print("layers:", list(adata.layers.keys()))
print(adata.obs["cell_type"].value_counts())
print("excluded cells:", int(adata.obs["exclude"].sum()))


## Cycling score
A 15-gene proliferation score. Cells with a score above 0.5 are treated as cycling in notebook 03. The threshold was chosen by eye from the histogram; the cell counts at other thresholds are printed for sensitivity.

In [ ]:
cyc_genes = ["MKI67", "TOP2A", "PCLAF", "TYMS", "RRM2", "CDK1", "BIRC5",
             "UBE2C", "TK1", "STMN1", "CDT1", "TPX2", "MYBL2", "CDCA7", "FAM111B"]
cyc_genes = [g for g in cyc_genes if g in adata.var_names]
sc.tl.score_genes(adata, cyc_genes, score_name="cycling_score")

print(adata.obs.groupby("cell_type", observed=True)["cycling_score"]
      .describe(percentiles=[.5, .9, .99]).round(2))

import matplotlib.pyplot as plt

fig_dir = out_dir / "figures"
tab_dir = out_dir / "tables"
fig_dir.mkdir(exist_ok=True)
tab_dir.mkdir(exist_ok=True)
sc.settings.figdir = fig_dir

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(adata.obs["cycling_score"], bins=100)
ax.set_yscale("log"); ax.set_xlabel("cycling score"); ax.set_ylabel("cells")
fig.savefig(fig_dir / "cycling_score_hist.png", dpi=200, bbox_inches="tight")
plt.close(fig)

# How sensitive is the cell count to the cutoff?
for t in [0.3, 0.4, 0.5, 0.6, 0.8]:
    print("score >", t, ":", int((adata.obs["cycling_score"] > t).sum()))

THR = 0.5
adata.obs["cycling"] = adata.obs["cycling_score"] > THR

print("\ncycling cells by cell type")
print(pd.crosstab(adata.obs["cell_type"], adata.obs["cycling"]))

x = adata.obs[adata.obs["cycling"] & ~adata.obs["exclude"]]
print("\ncycling cells by donor and day")
print(pd.crosstab(x["donor"], x["day"]))

sc.pl.umap(adata, color="cycling_score", vmax=1, save="_cycling_score.png")


## Save

In [ ]:
expected = {"CD4 T": 46149, "CD8 T": 27278, "NK": 14604, "Monocyte": 6879, "B": 6376,
            "Doublet-like": 2126, "T activated": 893, "Plasma": 659, "pDC": 487,
            "cDC": 454, "Platelet": 47}
got = adata.obs["cell_type"].value_counts().to_dict()
print("cell-type counts match the reported analysis:", got == expected)
if got != expected:
    print("differences:", {k: (got.get(k), expected.get(k)) for k in expected if got.get(k) != expected[k]})

adata.write_h5ad(out_dir / "annotated.h5ad", compression="gzip")
adata.obs[["donor", "day", "group", "batch", "leiden", "cell_subtype", "cell_type", "exclude"]].to_csv(
    tab_dir / "cell_metadata_annotated.csv")
print("saved annotated.h5ad")


## Optional: does the double normalisation matter?

In [ ]:
# OPTIONAL robustness check (a few minutes): re-cluster on the correctly log-normalised layer and
# compare with the clusters used for annotation. Adjusted Rand index near 1 = same partition.
from sklearn.metrics import adjusted_rand_score

chk = sc.AnnData(adata.layers["lognorm"].copy(),
                 obs=adata.obs[["donor"]].copy(),
                 var=pd.DataFrame(index=adata.var_names))
sc.pp.highly_variable_genes(chk, n_top_genes=3000, batch_key="donor")
sc.pp.pca(chk, n_comps=30, mask_var="highly_variable")
sc.pp.neighbors(chk, n_pcs=30)
sc.tl.leiden(chk, resolution=0.6, flavor="igraph", n_iterations=2, key_added="leiden_single")

ari = adjusted_rand_score(adata.obs["leiden"].astype(str), chk.obs["leiden_single"].astype(str))
tab = pd.crosstab(chk.obs["leiden_single"], adata.obs["cell_type"])
purity = tab.max(axis=1).sum() / tab.values.sum()
print("clusters (double-normalised run):", adata.obs["leiden"].nunique(),
      "| clusters (single normalisation):", chk.obs["leiden_single"].nunique())
print("adjusted Rand index between the two clusterings:", round(ari, 3))
print("share of cells whose new cluster is dominated by their assigned cell type:", round(purity, 3))
